This notebook demonstrates end‑to‑end deployment of a custom fraud‑latency model on Databricks using MLflow PyFunc and a Databricks Model Serving endpoint. The model connects to a Databricks‑hosted Postgres instance, queries specific row positions, computes an average score, and exposes that logic via a managed serving endpoint for real‑time inference.

In [0]:
##  Imports and global configuration (code cell) 
import os
import time
import psycopg2
import mlflow
import requests
import pandas as pd

from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from azure.identity import ClientSecretCredential

# ==========================================
# OIDC configuration (for DB access token)
# ==========================================
OIDC_TOKEN_URL = ""
CLIENT_ID = ""
CLIENT_SECRET = ""
SCOPES = ""

# ==========================================
# Database / model / endpoint configuration
# ==========================================
DB_HOST = ""
DB_NAME = ""

SCHEMA = ""
TABLE = ""
ORDER_COLUMN = ""
UC_MODEL_NAME = ""

# Serving endpoint configs
ENDPOINT_NAME = ""
WORKLOAD_SIZE = ""
MODEL_ALIAS = ""

# SPN secret scope for endpoint auth
SCOPE_NAME = ""
TENANT_ID = dbutils.secrets.get(SCOPE_NAME, "")
SPN_CLIENT_ID = dbutils.secrets.get(SCOPE_NAME, "")
SPN_CLIENT_SECRET = dbutils.secrets.get(SCOPE_NAME, "")

# Databricks workspace host (for REST calls)
HOST = ""


In [0]:
# =========================================================
# PyFunc fraud latency model implementation
# =========================================================

class FraudModelPOC(mlflow.pyfunc.PythonModel):
    """
    Custom MLflow PyFunc model that:
    - Authenticates via OIDC to get a Postgres OAuth token
    - Connects to the Databricks Postgres instance
    - Queries specific row positions using a window function
    - Returns average score and raw rows, plus latency metrics
    """

    def __init__(self, db_host, db_name, schema, table, order_column):
        self.db_host = db_host
        self.db_name = db_name
        self.schema = schema
        self.table = table
        self.order_column = order_column

    def load_context(self, context):
        """
        Called once when the model is loaded in the serving environment.
        Uses client credentials to obtain an access token for the database.
        """
        data = {
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "scope": SCOPES,
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        resp = requests.post(OIDC_TOKEN_URL, data=data, headers=headers, timeout=10)
        resp.raise_for_status()
        token = resp.json()["access_token"]

        self.oauth_token = token
        self.client_id = CLIENT_ID

    def predict(self, context, model_input):
        """
        Inference entry point used by MLflow PyFunc.
        Supports:
        - Pandas DataFrame with 'db_host' and 'positions' columns
        - dict with 'db_host' and 'positions' keys

        positions: list of row indices to fetch (1‑based rank over ORDER_COLUMN).
        """
        # Normalize inputs
        if isinstance(model_input, pd.DataFrame):
            db_host_val = str(model_input.loc[0, "db_host"])
            positions_val = model_input.loc[0, "positions"]
        elif isinstance(model_input, dict):
            db_host_val = model_input.get("db_host", self.db_host)
            positions_val = model_input.get("positions", [10, 30, 100, 400])
        else:
            raise TypeError(f"Unexpected input type: {type(model_input)}")

        latencies = {}

        # DB connection
        t0 = time.perf_counter()
        conn = psycopg2.connect(
            host=db_host_val,
            dbname=self.db_name,
            user=self.client_id,
            password=self.oauth_token,
            sslmode="require",
            connect_timeout=10,
        )
        latencies["connect_ms"] = round((time.perf_counter() - t0) * 1000, 2)

        # Query the requested positions using a window function
        query = f"""
            WITH numbered AS (
                SELECT
                    spider_tid,
                    t1_ts,
                    t2_ts,
                    latency_s,
                    ROW_NUMBER() OVER (ORDER BY {self.order_column} ASC) AS rn
                FROM {self.schema}.{self.table}
            )
            SELECT * FROM numbered WHERE rn = ANY(%s);
        """

        t1 = time.perf_counter()
        with conn.cursor() as cur:
            cur.execute(query, (positions_val,))
            rows = cur.fetchall()
        latencies["query_ms"] = round((time.perf_counter() - t1) * 1000, 2)

        # Compute average score (example: use second column)
        t2 = time.perf_counter()
        avg_score = sum([row[1] for row in rows]) / len(rows)
        latencies["calc_ms"] = round((time.perf_counter() - t2) * 1000, 2)

        conn.close()

        # Return structured JSON‑like result (compatible with serving)
        return {
            "predictions": {
                "avg_score": avg_score,
                "rows": rows,
            },
            "latencies_ms": latencies,
        }


In [0]:
# =========================================================
# Register FraudModelPOC into Unity Catalog via MLflow
# =========================================================

mlflow.set_experiment("/Shared/fraud_model_poc_experiment")

with mlflow.start_run():
    model = FraudModelPOC(
        db_host=DB_HOST,
        db_name=DB_NAME,
        schema=SCHEMA,
        table=TABLE,
        order_column=ORDER_COLUMN,
    )

    # Call load_context once so we can build an example output
    model.load_context(None)

    # Example input/output for signature inference (no live DB call)
    input_example = {"db_host": DB_HOST, "positions": [10, 30]}
    example_output = {
        "predictions": {
            "avg_score": 0.85,
            "rows": [
                ("tx123", 0.85, "2025-12-08T00:00:00Z", 12.5, 1),
                ("tx124", 0.90, "2025-12-08T00:01:00Z", 13.1, 2),
            ],
        },
        "latencies_ms": {
            "connect_ms": 10.0,
            "query_ms": 25.0,
            "calc_ms": 5.0,
        },
    }

    # Infer model signature from the mocked input/output
    signature = infer_signature(input_example, example_output)

    # Log and register PyFunc model to Unity Catalog
    mlflow.pyfunc.log_model(
        artifact_path="fraud_model_poc",
        python_model=model,
        registered_model_name=UC_MODEL_NAME,
        signature=signature,
        pip_requirements=["mlflow", "psycopg2-binary", "requests", "pandas"],
    )

print(f"✅ Model registered in Unity Catalog with signature at {UC_MODEL_NAME}")


In [0]:
# =========================================================
# Resolve latest model version and fetch Databricks token
# =========================================================

client = MlflowClient()
versions = [int(m.version) for m in client.search_model_versions(f"name='{UC_MODEL_NAME}'")]
latest_version = str(max(versions))
print(f"ℹ️ Latest version for {UC_MODEL_NAME} is {latest_version}")

# Use Azure service principal to obtain a Databricks workspace token
cred = ClientSecretCredential(
    tenant_id=TENANT_ID,
    client_id=SPN_CLIENT_ID,
    client_secret=SPN_CLIENT_SECRET,
)

db_token = cred.get_token("").token

headers = {
    "Authorization": f"Bearer {db_token}",
    "Content-Type": "application/json",
}


In [0]:
# =========================================================
# Create or update Databricks Model Serving endpoint
# =========================================================

def create_or_update_serving_endpoint(
    model_name,
    model_version,
    model_alias,
    workload_size="GPU small",
):
    payload = {
        "name": ENDPOINT_NAME,
        "config": {
            "served_models": [
                {
                    "model_name": model_name,
                    "model_version": str(model_version),
                    "model_alias": model_alias,
                    "workload_type": "GPU small",
                    "workload_size": workload_size,
                }
            ]
        },
    }

    url = f"{HOST}api/2.0/serving-endpoints"
    r = requests.post(url, headers=headers, json=payload)

    if r.status_code == 200:
        print(f"✅ Serving endpoint '{ENDPOINT_NAME}' created")
    elif r.status_code == 409:
        print(f"ℹ️ Endpoint '{ENDPOINT_NAME}' exists; updating config...")
        update_payload = payload["config"]
        u = requests.put(
            f"{HOST}api/2.0/serving-endpoints/{ENDPOINT_NAME}/config",
            headers=headers,
            json=update_payload,
        )
        if u.ok:
            print("✅ Endpoint updated (or config unchanged)")
        else:
            print("❌ Endpoint update failed:", u.text)
            raise SystemExit()
    else:
        print("❌ Endpoint creation failed:", r.text)
        raise SystemExit(f"Failed to create or update endpoint: {r.text}")

create_or_update_serving_endpoint(
    model_name=UC_MODEL_NAME,
    model_version=latest_version,
    model_alias=MODEL_ALIAS,
    workload_size=WORKLOAD_SIZE,
)
